# YouTube Downloader & Metadata Extractor (Colab)
Đây là script tải video và trích xuất metadata (fps, codec) mô phỏng logic trong thư mục `src/download`.
**Quy trình:**
1. Hãy chuẩn bị file CSV đầu vào (ví dụ `videos.csv`) chứa 1 cột có tên là `video_id`.
2. Upload file CSV đó lên thư mục làm việc hiện tại của Colab.
3. Chạy từng Cell bên dưới.

In [ ]:
# Cài đặt yt-dlp, cài pandas (có sẵn trên Colab, cài phòng hờ) và ffmpeg cho colab
!pip install yt-dlp pandas tqdm
!apt-get update -qq && apt-get install -y ffmpeg

In [ ]:
import os
import json
import subprocess
from pathlib import Path

import pandas as pd
import numpy as np
import yt_dlp
from tqdm.auto import tqdm

# ==========================================
# CẤU HÌNH ĐƯỜNG DẪN
# ==========================================
INPUT_CSV = "videos.csv"  # Tên file gốc mà bạn đã upload
OUTPUT_DIR = Path("downloaded_videos")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def run_ffprobe(video_path):
    """Chạy ffprobe để lấy thông tin stream của video (Dựa trên ffprobe_extract.py)."""
    cmd = [
        "ffprobe",
        "-v", "error",
        "-show_streams",
        "-show_format",
        "-of", "json",
        str(video_path),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)
    return json.loads(result.stdout)

def get_fps_and_codec(video_path):
    """Trích xuất fps và codec từ json metadata trả về bởi ffprobe."""
    try:
        data = run_ffprobe(video_path)
        video_stream = next((s for s in data.get("streams", []) if s.get("codec_type") == "video"), None)
        if not video_stream:
            return None, None
            
        # Tính FPS (Ví dụ chuỗi là 30000/1001)
        fps_str = video_stream.get("avg_frame_rate") or video_stream.get("r_frame_rate")
        fps = None
        if fps_str:
            if "/" in fps_str:
                num, den = fps_str.split("/")
                fps = float(num) / float(den) if float(den) != 0 else None
            else:
                fps = float(fps_str)
                
        # Lấy Codec Name (vd: h264)
        codec = video_stream.get("codec_name")
        return fps, codec
    except Exception as e:
        print(f"Lỗi ffprobe: {e}")
        return None, None

def build_format_selector(download_mode: str = "720p_mp4") -> str:
    """Dựa trên youtube_download.py để lấy mp4 nhỏ hơn hoặc bằng 720p"""
    if download_mode == "best":
         return "bv*+ba/b"
    if download_mode == "720p_mp4":
         return "bv*[ext=mp4][height<=720]+ba[ext=m4a]/b[ext=mp4][height<=720]/b"
    return "bv*+ba/b"


In [ ]:
# ==========================================
# XỬ LÝ TẢI VIDEO & METADATA VÀ LƯU CSV
# ==========================================
if not os.path.exists(INPUT_CSV):
    print(f"LỖI: Không tìm thấy file {INPUT_CSV}. Vui lòng upload lên Colab!")
else:
    df = pd.read_csv(INPUT_CSV)
    
    # Format url dựa theo video_id
    if 'url' not in df.columns:
        df['url'] = "https://www.youtube.com/watch?v=" + df['video_id'].astype(str)
        
    # Khởi tạo các cột với Dtype chính xác để tránh cảnh báo (FutureWarning) của Pandas
    for col in ['title', 'codec', 'status']:
        if col not in df.columns:
            df[col] = ""
        df[col] = df[col].astype('object')
        
    if 'fps' not in df.columns:
        df['fps'] = np.nan
    df['fps'] = df['fps'].astype('float64')

    # Cấu hình downloader thông qua yt-dlp
    ydl_opts = {
        'outtmpl': str(OUTPUT_DIR / '%(id)s.%(ext)s'),
        'format': build_format_selector("720p_mp4"),
        'noplaylist': True,
        'quiet': True,
        'no_warnings': True,
    }

    # Duyệt file csv
    for index, row in tqdm(df.iterrows(), total=len(df), desc="Đang xử lý Video"):
        video_id = str(row['video_id']).strip()
        url = row['url']
        
        # 1. Kiểm tra nếu video đã được tải về (chặn sớm để tiết kiệm thời gian)
        existing_files = list(OUTPUT_DIR.glob(f"{video_id}.*"))
        if existing_files:
            downloaded_path = existing_files[0]
            fps, codec = get_fps_and_codec(downloaded_path)
            df.at[index, 'fps'] = round(fps, 2) if fps else None
            df.at[index, 'codec'] = codec if codec else ""
            df.at[index, 'status'] = 'skipped_exists'
            continue
            
        # 2. Nếu chưa tồn tại, tiến hành gọi yt-dlp để tải
        try:
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                # Tải file từ youtube
                info = ydl.extract_info(url, download=True)
                
                df.at[index, 'title'] = info.get('title') if info.get('title') else ""
                ext = info.get('ext', 'mp4')
                downloaded_path = OUTPUT_DIR / f"{video_id}.{ext}"
                
                # Dùng thuật toán lấy info ffprobe để tính fps, codec cho file gốc
                if downloaded_path.exists():
                    fps, codec = get_fps_and_codec(downloaded_path)
                    df.at[index, 'fps'] = round(fps, 2) if fps else None
                    df.at[index, 'codec'] = codec if codec else ""
                    df.at[index, 'status'] = 'ok'
                else:
                    df.at[index, 'status'] = 'file_not_found'
                     
        except Exception as e:
            # Những video Private/Bị xoá/Age restricted sẽ nhảy vào đây
            print(f"Lỗi ở video {video_id}: {e}")
            df.at[index, 'status'] = 'error'

    # Lưu giữ lại các cột này: video_id, url, title, fps, codec, status vào csv gốc
    columns_to_save = ['video_id', 'url', 'title', 'fps', 'codec', 'status']
    df = df[columns_to_save]
    df.to_csv(INPUT_CSV, index=False)
    print(f"\nHoàn tất! Danh sách kết quả và metadata đã được lưu đè lại vào {INPUT_CSV}")
    display(df.head())
